<a href="https://colab.research.google.com/github/giannismantzaris-cmd/DAMA61/blob/main/Mantzaris%20-%20DAMA61_final2026_problem3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h2 style="text-align:center;"> HELLENIC OPEN UNIVERSITY - SCHOOL OF SCIENCE AND TECHNOLOGY</h2>
<h2 style="text-align:center;"> DATA SCIENCE AND MACHINE LEARNING : DAMA61 ACAD. YEAR 2025-26</h2>
<h3 style="text-align:center;"> FINAL EXAM PROBLEM 3</h3>
<hr>

## Problem 3

Analyze the Fashion MNIST dataset using a stacked autoencoder and apply transfer learning by reusing the encoder part of the trained autoencoder for image classification. Set random seed for TensorFlow and NumPy to 42 to ensure reproducibility.

1. Load the Fashion MNIST dataset. From the loaded training set retain a stratified subset of <b>20,000 samples</b> together with their corresponding labels, ensuring that all 10 classes are represented in the subset. Flatten the 28x28 images into 784-dimensional vectors, scale the images to the interval [0,1], and split the dataset into training, evaluation and testing sets, using 70% of the samples for training, 10% for validation and 20% for testing. Transform the target labels appropriately. <b>[10%]</b>
</br>

2. Build a Stacked Autoencoder. The encoder should consist of <b>[25%]</b>:
   <ul>
      <li>a dense hidden layer with 128 nodes and ReLU activation,</li>
      <li>a dense hidden layer with 64 nodes and ReLU activation,</li>
      <li>a coding layer with 32 nodes and ReLU activation.</li>
   </ul>

   The decoder should be symmetric to the encoder and consist of:
   <ul>
      <li>a dense hidden layer with 64 nodes and ReLU activation,</li>
      <li>a dense hidden layer with 128 nodes and ReLU activation,</li>
      <li>an output layer with 784 nodes and ReLU activation.</li>
   </ul>
   
</br>
3. Compile the autoencoder model using the Adam optimizer and MSE loss function. Allow the model to train for 20 epochs with a batch size of 32. <b>[15%]</b>
</br>

4. After training the autoencoder, extract the encoder part of the model. The encoder should take the original 784-dimensional image vectors as input and produce the 32-dimensional encoded representations. Apply transfer learning by creating a new classification model that reuses the trained encoder as its first part. Freeze the weights of all encoder layers so they are not updated during the initial classification training. Attach the following classification layers after the frozen encoder <b>[20%]</b>:
   <ul>
      <li>a dense hidden layer with 24 nodes and ReLU activation,</li>
      <li>a dense hidden layer with 16 nodes and ReLU activation,</li>
      <li>a final output layer with 10 nodes and softmax activation, corresponding to the 10 Fashion MNIST classes.</li>
   </ul>
   
</br>
5. Compile the classification model using the Adam optimizer and categorical cross-entropy loss function. Train the model for 10 epochs with a batch size of 32, updating only the newly added classification layers while keeping the encoder frozen. Evaluate the classification model on the test set and report the test accuracy. Fine tune the model by unfreezing the encoder weights and training for an additional 10 epochs. Briefly comment on how the pretrained encoder is used for transfer learning in this classification task. <b>[30%]</b>

In [12]:
# import the needed packages

import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense

np.random.seed(42)
tf.random.set_seed(42)

In [4]:
# load the Fashion-MNIST dataset

# Load Fashion MNIST
(x_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

print(x_train_full.shape)
print(y_train_full.shape)

(60000, 28, 28)
(60000,)


In [5]:
# keep a stratified subset of 20.000 samples. the rest we will not use

X_subset, X_not_used, y_subset, y_not_used = train_test_split(
    x_train_full,
    y_train_full,
    train_size=20000,
    stratify=y_train_full,
    random_state=42
)

In [6]:
# flatten and normalize

X_subset = X_subset.reshape(-1, 28 * 28)
X_subset = X_subset.astype("float32") / 255.0

In [8]:
# split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_subset,
    y_subset,
    train_size=0.70,
    stratify=y_subset,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    stratify=y_temp,
    random_state=42
)

# Verify shapes

print("Training set:  ", X_train.shape, y_train.shape)
print("Validation set:", X_valid.shape, y_valid.shape)
print("Test set:      ", X_test.shape, y_test.shape)

Training set:   (14000, 784) (14000,)
Validation set: (2000, 784) (2000,)
Test set:       (4000, 784) (4000,)


In [9]:
# Transform the target labels appropriately

y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_valid = tf.keras.utils.to_categorical(y_valid, num_classes=10)
y_test  = tf.keras.utils.to_categorical(y_test, num_classes=10)

In [10]:
# print shapes
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)
print("y_test :", y_test.shape)

y_train: (14000, 10)
y_valid: (2000, 10)
y_test : (4000, 10)


In [13]:
# build the Stacked Autoencoder

stacked_autoencoder = Sequential()
stacked_autoencoder.add(InputLayer(shape=(784,)))
stacked_autoencoder.add(Dense(128, activation="relu"))
stacked_autoencoder.add(Dense(64, activation="relu"))
stacked_autoencoder.add(Dense(32, activation="relu"))

stacked_autoencoder.add(Dense(64, activation="relu"))
stacked_autoencoder.add(Dense(128, activation="relu"))
stacked_autoencoder.add(Dense(784, activation="relu"))

In [14]:
# display model summary

stacked_autoencoder.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 784)            │       101,136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 222,384 (868.69 KB)

 Trainable params: 222,384 (868.69 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# compile and train the stacked autoencoder

stacked_autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

history_autoencoder = stacked_autoencoder.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_valid, X_valid)
)

Epoch 1/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0520 - val_loss: 0.0341
Epoch 2/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0300 - val_loss: 0.0273
Epoch 3/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0267 - val_loss: 0.0253
Epoch 4/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0244 - val_loss: 0.0231
Epoch 5/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0227 - val_loss: 0.0219
Epoch 6/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0215 - val_loss: 0.0209
Epoch 7/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0206 - val_loss: 0.0205
Epoch 8/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0201 - val_loss: 0.0201
Epoch 9/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0197 - val_loss: 0.0196
Epoch 10/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0194 - val_loss: 0.0194
Epoch 11/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0192 - val_loss: 0.0192
Epoch 12/20
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/ste

In [16]:
# extract the encoder part of the autoencoder

encoder = Sequential()

encoder.add(InputLayer(shape=(784,)))
encoder.add(stacked_autoencoder.layers[0])
encoder.add(stacked_autoencoder.layers[1])
encoder.add(stacked_autoencoder.layers[2])

encoder.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,816 (432.88 KB)

 Trainable params: 110,816 (432.88 KB)

 Non-trainable params: 0 (0.00 B)